# ARC-CAPTCHA: Human vs Bot Behavior Analysis

This notebook explores behavioral features from ARC-CAPTCHA sessions to understand
how human and bot interactions differ. It loads session data, computes features,
visualizes distributions, and evaluates classifier accuracy.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Session Data

Load from JSON files generated by the bot simulator and (eventually) from Supabase.

In [ ]:
DATA_DIR = Path("../data")


def load_sessions_from_dir(data_dir: Path) -> list[dict]:
    """Load all session JSON files from a directory."""
    sessions = []
    combined = data_dir / "all_bot_sessions.json"
    if combined.exists():
        with open(combined) as f:
            sessions.extend(json.load(f))
        return sessions

    for filepath in sorted(data_dir.glob("*.json")):
        with open(filepath) as f:
            data = json.load(f)
            if isinstance(data, list):
                sessions.extend(data)
            else:
                sessions.append(data)
    return sessions


sessions = load_sessions_from_dir(DATA_DIR)
print(f"Loaded {len(sessions)} sessions")
if sessions:
    print(f"Strategies: {set(s.get('strategy', 'unknown') for s in sessions)}")

## 2. Feature Extraction

Compute the same 5 behavioral features as the TypeScript classifier:
- Action interval variance
- Exploration diversity
- Undo ratio
- Time to first action
- Action entropy

In [ ]:
def compute_interval_variance(logs: list[dict]) -> float:
    """Variance of time intervals between actions (normalized 0-1)."""
    if len(logs) < 2:
        return 0.0
    intervals = [
        log["timeSinceLastAction"]
        for log in logs[1:]
        if log["timeSinceLastAction"] > 0
    ]
    if not intervals:
        return 0.0
    variance = np.var(intervals)
    return min(1.0, float(1 - 1 / (1 + variance / 5000)))


def compute_exploration_diversity(logs: list[dict]) -> float:
    """Unique cells or frames / total actions."""
    if not logs:
        return 0.0
    coords = set()
    for log in logs:
        if log.get("coordinates"):
            c = log["coordinates"]
            coords.add((c[0], c[1]))
    if not coords:
        unique_frames = len(set(log["frameHash"] for log in logs))
        return min(1.0, unique_frames / max(1, len(logs)))
    return min(1.0, len(coords) / len(logs))


def compute_undo_ratio(logs: list[dict]) -> float:
    """Undo actions / total actions (normalized)."""
    if not logs:
        return 0.0
    undo_count = sum(1 for log in logs if log["actionType"] == "undo")
    ratio = undo_count / len(logs)
    if ratio == 0:
        return 0.0
    if ratio > 0.5:
        return 0.3
    return min(1.0, ratio / 0.1)


def compute_time_to_first_action(logs: list[dict]) -> float:
    """Time before first action (normalized 0-1)."""
    if not logs:
        return 0.0
    first = logs[0].get("timeSinceLastAction", 0)
    deliberation = first if first > 0 else (logs[1]["timeSinceLastAction"] if len(logs) > 1 else 0)
    if deliberation < 50:
        return 0.0
    if deliberation > 10000:
        return 0.5
    return min(1.0, deliberation / 1500)


def compute_action_entropy(logs: list[dict]) -> float:
    """Shannon entropy of action type distribution (normalized 0-1)."""
    if not logs:
        return 0.0
    counts: dict[str, int] = {}
    for log in logs:
        t = log["actionType"]
        counts[t] = counts.get(t, 0) + 1
    total = len(logs)
    entropy = 0.0
    for count in counts.values():
        p = count / total
        if p > 0:
            entropy -= p * math.log2(p)
    max_entropy = math.log2(4)
    return min(1.0, entropy / max_entropy)


def extract_features(logs: list[dict]) -> dict[str, float]:
    """Extract all 5 behavioral features."""
    return {
        "interval_variance": compute_interval_variance(logs),
        "exploration_diversity": compute_exploration_diversity(logs),
        "undo_ratio": compute_undo_ratio(logs),
        "time_to_first_action": compute_time_to_first_action(logs),
        "action_entropy": compute_action_entropy(logs),
    }


# Classifier weights (matching TypeScript implementation)
WEIGHTS = {
    "interval_variance": 0.25,
    "exploration_diversity": 0.15,
    "undo_ratio": 0.15,
    "time_to_first_action": 0.20,
    "action_entropy": 0.25,
}


def classify(features: dict[str, float]) -> tuple[bool, float]:
    """Rule-based classifier. Returns (is_human, confidence)."""
    score = sum(features[k] * WEIGHTS[k] for k in WEIGHTS)
    is_human = score > 0.4
    confidence = min(1.0, abs(score - 0.4) / 0.4)
    return is_human, confidence

## 3. Compute Features for All Sessions

In [ ]:
rows = []
for session in sessions:
    logs = session.get("actionLog", [])
    features = extract_features(logs)
    is_human, confidence = classify(features)

    rows.append(
        {
            "session_id": session["sessionId"],
            "strategy": session.get("strategy", "unknown"),
            "source": session.get("source", "unknown"),
            "actual_human": session.get("isHuman", False),
            "predicted_human": is_human,
            "confidence": confidence,
            "action_count": session.get("actionCount", len(logs)),
            **features,
        }
    )

df = pd.DataFrame(rows)
print(f"Feature matrix: {df.shape}")
df.head(10)

## 4. Feature Distributions: Human vs Bot

In [ ]:
feature_cols = [
    "interval_variance",
    "exploration_diversity",
    "undo_ratio",
    "time_to_first_action",
    "action_entropy",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    for label in df["source"].unique():
        subset = df[df["source"] == label]
        ax.hist(subset[col], bins=20, alpha=0.5, label=label)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
    ax.legend()

# Hide extra subplot
axes[-1].set_visible(False)
plt.suptitle("Feature Distributions by Source", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=ax, fmt=".2f")
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 6. Classifier Accuracy

Evaluate how well the rule-based classifier separates humans from bots.

In [ ]:
if len(df) > 0:
    y_true = df["actual_human"].astype(int)
    y_pred = df["predicted_human"].astype(int)

    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["Bot", "Human"]))

    print(f"\nAccuracy: {accuracy_score(y_true, y_pred):.2%}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Bot", "Human"],
        yticklabels=["Bot", "Human"],
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("No sessions loaded. Run bot_simulator.py first.")

## 7. Per-Strategy Analysis

In [ ]:
if "strategy" in df.columns and len(df) > 0:
    strategy_stats = df.groupby("strategy")[feature_cols].agg(["mean", "std"])
    print("Feature Statistics by Strategy:")
    print(strategy_stats.round(3).to_string())

    # Box plot per strategy
    fig, axes = plt.subplots(1, len(feature_cols), figsize=(20, 5))
    for i, col in enumerate(feature_cols):
        df.boxplot(column=col, by="strategy", ax=axes[i])
        axes[i].set_title(col.replace("_", " ").title())
        axes[i].set_xlabel("")
    plt.suptitle("Feature Distributions by Bot Strategy", fontsize=14)
    plt.tight_layout()
    plt.show()

## 8. Summary

Key observations and next steps:

1. **Interval variance** is the strongest discriminator — bots have very regular timing
2. **Action entropy** captures diversity of action types
3. **Undo ratio** is useful but only when humans actually undo
4. **Time to first action** helps detect instant-start bots
5. **Exploration diversity** differentiates random exploration from systematic patterns

### Next steps:
- Collect real human sessions via the demo app
- Compare real human data against simulated bot data
- Tune classifier weights based on ROC/AUC analysis
- Consider ML classifier (e.g., logistic regression, random forest) for v2